In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
url = "https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main"

resp = requests.get(url)

print(resp.status_code)

200


In [3]:
zf = zipfile.ZipFile(io.BytesIO(resp.content))

len(zf.infolist())

1490

In [4]:
repository_data = []

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    if not filename.endswith(".md"):
        continue

    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")

        post = frontmatter.loads(content)

        data = post.to_dict()

        data["filename"] = filename

        repository_data.append(data)

In [5]:
len(repository_data)

1216

In [6]:
repository_data[0]

{'description': 'Review and process open FAQ PRs',
 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -

In [7]:
def read_repo_data(repo_owner, repo_name):

    prefix = "https://codeload.github.com"

    url = f"{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main"

    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception("Download failed")

    repository_data = []

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():

        filename = file_info.filename
        filename_lower = filename.lower()

        if not (
            filename_lower.endswith(".md")
            or filename_lower.endswith(".mdx")
        ):
            continue

        try:
            with zf.open(file_info) as f_in:

                content = f_in.read().decode(
                    "utf-8",
                    errors="ignore"
                )

                post = frontmatter.loads(content)

                data = post.to_dict()

                data["filename"] = filename

                repository_data.append(data)

        except Exception as e:
            print(e)

    zf.close()

    return repository_data

In [8]:
faq_data = read_repo_data(
    "DataTalksClub",
    "faq"
)

len(faq_data)

1216

In [10]:
my_repo = read_repo_data(
    "ezhilarasi0509",
    "AI-Agent-"
)

len(my_repo)

1

In [12]:
evidently_docs = read_repo_data(
    "evidentlyai",
    "docs"
)

len(evidently_docs)

95

In [13]:
evidently_docs[45]

{'title': 'LLM regression testing',
 'description': 'How to run regression testing for LLM outputs.',
 'content': 'In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You\'ll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere\'s what we\'ll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to eval

In [14]:
len(evidently_docs[45]["content"])

21712

In [15]:
def sliding_window(seq, size, step):

    if size <= 0 or step <= 0:
        raise ValueError(
            "size and step must be positive"
        )

    n = len(seq)

    result = []

    for i in range(0, n, step):

        chunk = seq[i:i + size]

        result.append({
            "start": i,
            "chunk": chunk
        })

        if i + size >= n:
            break

    return result

In [16]:
text = evidently_docs[45]["content"]

chunks = sliding_window(
    text,
    2000,
    1000
)

len(chunks)

21

In [17]:
chunks[0]

{'start': 0,
 'chunk': "In this tutorial, you will learn how to perform regression testing for LLM outputs.\n\nYou can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.\n\n<Info>\n  **This example uses Evidently Cloud.** You'll run evals in Python and upload them. You can also skip the upload and view Reports locally. For self-hosted, replace `CloudWorkspace` with `Workspace`.\n</Info>\n\n# Tutorial scope\n\nHere's what we'll do:\n\n* **Create a toy dataset**. Build a small Q&A dataset with answers and reference responses.\n\n* **Get new answers**. Imitate generating new answers to the same question.\n\n* **Create and run a Report with Tests**. Compare the answers using LLM-as-a-judge to evaluate length, correctness and style consistency.\n\n* **Build a monitoring Dashboard**. Get plo

In [18]:
evidently_chunks = []

for doc in evidently_docs:

    doc_copy = doc.copy()

    doc_content = doc_copy.pop("content")

    chunks = sliding_window(
        doc_content,
        2000,
        1000
    )

    for chunk in chunks:

        chunk.update(doc_copy)

    evidently_chunks.extend(chunks)

In [19]:
len(evidently_chunks)

573

In [20]:
evidently_chunks[0]

{'start': 0,
 'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
 'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx'}

In [21]:
import json

with open(
    "evidently_chunks.json",
    "w"
) as f:

    json.dump(
        evidently_chunks,
        f,
        indent=2
    )

In [22]:
import re

text = evidently_docs[45]["content"]

paragraphs = re.split(
    r"\n\s*\n",
    text.strip()
)

len(paragraphs)

153

In [23]:
paragraphs[0]

'In this tutorial, you will learn how to perform regression testing for LLM outputs.'

In [24]:
import re

def split_markdown_by_level(
    text,
    level=2
):

    header_pattern = (
        r'^(#{'
        + str(level)
        + r'} )(.+)$'
    )

    pattern = re.compile(
        header_pattern,
        re.MULTILINE
    )

    parts = pattern.split(text)

    sections = []

    for i in range(1, len(parts), 3):

        header = (
            parts[i]
            + parts[i + 1]
        )

        header = header.strip()

        content = ""

        if i + 2 < len(parts):

            content = parts[i + 2].strip()

        if content:

            section = (
                f"{header}\n\n{content}"
            )

        else:
            section = header

        sections.append(section)

    return sections

In [25]:
sections = split_markdown_by_level(
    text,
    level=2
)

len(sections)

8

In [26]:
sections[0]

'## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.dash

In [27]:
section_chunks = []

for doc in evidently_docs:

    doc_copy = doc.copy()

    doc_content = doc_copy.pop(
        "content"
    )

    sections = split_markdown_by_level(
        doc_content,
        level=2
    )

    for section in sections:

        section_doc = doc_copy.copy()

        section_doc["section"] = section

        section_chunks.append(
            section_doc
        )

In [28]:
len(section_chunks)

264

In [29]:
section_chunks[0]

{'title': 'Introduction',
 'description': 'Example section for showcasing API endpoints',
 'filename': 'docs-main/api-reference/introduction.mdx',
 'section': '## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>'}

In [30]:
with open(
    "section_chunks.json",
    "w"
) as f:

    json.dump(
        section_chunks,
        f,
        indent=2
    )